# Validate linear summation (metamodel vs dynamic)

Compare offshore-driven **dynamic SnapWave** to the **reconstructed metamodel**.

- **`mm.reconstruct(..., spectral=True)`** — partitions ``0``–``3`` per goal, wavespectra linear sum (validation vs dynamic).
- **`mm.reconstruct(..., spectral=False)`** — bulk ``hs/tp/dir/spr`` per goal (partition ``-1``), same wavespectra sum.

**Prerequisites**

1. Models fitted — `06_Validate_Metamodel.ipynb` (`mm.fit_all`)
2. Dynamic run — `08_Dynamic_Cases_Generation.ipynb` → joined NetCDF

Validation loads saved models only (no refitting).

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr
from utils import metamodel as mm
from utils import validation as val
from utils.config import SIMULATIONS, resolve

## Configuration

In [ ]:
period = "1989-09"
offshore_name = "WHACS"
mda_n = 450  # must match a fitted mda_candidates entry
pca_variance = {"hs": 0.99, "dir": 0.99}  # or 0.99, or {"hs": 0.99, "dir": 0.95}

# Site subset — isobath depth, mesh polygon, buoy id, or umbrella
sites = ["10m"]  # e.g. "10m", "buoy", ["10m", "buoy_Apolima_Buoy"]

dynamic_path = (
    resolve(SIMULATIONS["dynamic_cases"]) / f"Dynamic_Cases_{offshore_name}_fullaso.nc"
)

print(f"period       : {period}")
print(f"mda_n        : {mda_n}")
print(f"sites        : {sites}")
print(f"dynamic path : {dynamic_path}")

## 1 · Check models

In [ ]:
mm.require_models_all(mda_n, pca_variance=pca_variance)
print(f"common model sites: {len(mm.common_model_sites())}")

## 2 · Reconstruct metamodel (spectral)

In [ ]:
metamodel = mm.reconstruct(
    period,
    mda_n,
    spectral=True,
    offshore_name=offshore_name,
    sites=sites,
    pca_variance=pca_variance,
)
metamodel

## 3 · Load dynamic reference

In [ ]:
dynamic = xr.open_dataset(dynamic_path).sel(time=period)
site_ids = mm.resolve_reconstruction_sites(dynamic, sites)
dynamic = dynamic.sel(sites=site_ids).load()
dynamic

## 4 · Compare

In [ ]:
result = val.compare(
    metamodel.isel(time=slice(1, None))[["hs", "tp", "dir"]],
    dynamic[["hs", "tp", "dir"]],
    estimate_label="Linear summation",
    reference_label="Dynamic SnapWave",
)
result.metrics

In [ ]:
result.plot_all()
plt.show()